In [3]:
import numpy as np
import pandas as pd
from pathlib import Path

# --- Paths ---
OUTPUT_DIR = Path("/mnt/movement/users/jaizor/xtra/data/fmri/ucla")
fmri_file = OUTPUT_DIR / "fmri_ucla.npz"
pheno_file = OUTPUT_DIR / "pheno_ucla.csv"

# --- Load data ---
print("🔍 Loading parcellated fMRI data...")
data = np.load(fmri_file, allow_pickle=True)
X = data['data']          # Shape: (N, 150, 414)
subject_ids = data['subject_ids']

print("📋 Loading phenotype...")
pheno = pd.read_csv(pheno_file)

# --- Basic integrity checks ---
print(f"\n✅ fMRI array shape: {X.shape}")
print(f"✅ Subject IDs count: {len(subject_ids)}")
print(f"✅ Phenotype rows: {len(pheno)}")

assert X.shape[0] == len(subject_ids), "Mismatch: fMRI subjects vs subject_ids"
assert X.shape[0] == len(pheno), "Mismatch: fMRI subjects vs phenotype rows"
assert X.shape[1] == 150, f"Timepoints ≠ 150 → {X.shape[1]}"
assert X.shape[2] == 414, f"ROIs ≠ 414 → {X.shape[2]}"

# --- Check subject ID alignment ---
pheno_eids = pheno['eid'].astype(str).values
assert np.array_equal(subject_ids, pheno_eids), "Subject ID order mismatch!"

# --- Check data type and value range ---
print(f"\n📊 Data type: {X.dtype}")
print(f"📈 Value range: [{X.min():.3f}, {X.max():.3f}]")
print(f"🧮 Mean ± std per ROI (first 5):")
for i in range(5):
    roi_mean = X[:, :, i].mean()
    roi_std = X[:, :, i].std()
    print(f"  ROI {i+1:3d}: {roi_mean:+.4f} ± {roi_std:.4f}")

# --- Diagnosis distribution ---
if 'control' in pheno.columns:
    diag_cols = ['control', 'schz', 'bipolar', 'adhd']
    print(f"\n Diagnosis distribution:")
    for col in diag_cols:
        if col in pheno.columns:
            count = pheno[col].sum()
            print(f"  {col.upper():8s}: {int(count)}")

# --- Final confirmation ---
print(f"\n🎉 SUCCESS: All checks passed!")
print(f"   → Ready for modeling with {X.shape[0]} subjects, (150, 414) time series.")

🔍 Loading parcellated fMRI data...
📋 Loading phenotype...

✅ fMRI array shape: (132, 150, 414)
✅ Subject IDs count: 132
✅ Phenotype rows: 132

📊 Data type: float32
📈 Value range: [-11.237, 10.673]
🧮 Mean ± std per ROI (first 5):
  ROI   1: -0.0000 ± 1.0000
  ROI   2: +0.0000 ± 1.0000
  ROI   3: -0.0000 ± 1.0000
  ROI   4: -0.0000 ± 1.0000
  ROI   5: +0.0000 ± 1.0000

 Diagnosis distribution:
  CONTROL : 69
  SCHZ    : 15
  BIPOLAR : 22
  ADHD    : 26

🎉 SUCCESS: All checks passed!
   → Ready for modeling with 132 subjects, (150, 414) time series.


In [3]:
import pandas as pd



# Correct paths as used in your pipeline
adhd_df = pd.read_csv('/mnt/movement/users/jaizor/xtra/data/fmri/adhd/pheno_ADHD_all_runs.csv')
ucla_df = pd.read_csv('/mnt/movement/users/jaizor/xtra/data/fmri/ucla/pheno_ucla.csv')


# --- ADHD-200 ---
print("=== ADHD-200 Cohort ===")
print(f"Total subjects: {len(adhd_df)}")
print(f"Cases (ADHD=1): {adhd_df['ADHD'].sum()}")
print(f"Controls (ADHD=0): {(adhd_df['ADHD'] == 0).sum()}")

# Age by group
age_by_adhd = adhd_df.groupby('ADHD')['Age'].agg(['mean', 'std', 'count'])
print("\nAge (mean ± std):")
for label, row in age_by_adhd.iterrows():
    grp = "Cases" if label == 1 else "Controls"
    print(f"  {grp}: {row['mean']:.2f} ± {row['std']:.2f} (n={int(row['count'])})")

# Sex by group (Sex: 0 = female, 1 = male — inferred from samples)
sex_counts = adhd_df.groupby('ADHD')['Sex'].value_counts().unstack(fill_value=0)
sex_counts.columns = ['Female (Sex=0)', 'Male (Sex=1)']
print("\nSex distribution:")
print(sex_counts)

# --- UCLA ---
print("\n\n=== UCLA Clinical Cohort ===")
# Recode diagnosis: disorder = 1 if any of schz, bipolar, or adhd == 1
ucla_df['Diagnosis'] = ucla_df[['schz', 'bipolar', 'adhd']].max(axis=1)
print(f"Total subjects: {len(ucla_df)}")
print(f"Cases (any disorder): {ucla_df['Diagnosis'].sum()}")
print(f"Controls: {(ucla_df['Diagnosis'] == 0).sum()}")

# Age by group
age_by_ucla = ucla_df.groupby('Diagnosis')['Age'].agg(['mean', 'std', 'count'])
print("\nAge (mean ± std):")
for label, row in age_by_ucla.iterrows():
    grp = "Cases" if label == 1 else "Controls"
    print(f"  {grp}: {row['mean']:.2f} ± {row['std']:.2f} (n={int(row['count'])})")

# Sex by group
sex_ucla = ucla_df.groupby('Diagnosis')['Sex'].value_counts().unstack(fill_value=0)
sex_ucla.columns = ['Female (Sex=0)', 'Male (Sex=1)']
print("\nSex distribution:")
print(sex_ucla)

# Optional: Breakdown by specific disorder
print("\nDisorder-specific counts:")
print(f"  Schizophrenia: {ucla_df['schz'].sum()}")
print(f"  Bipolar:       {ucla_df['bipolar'].sum()}")
print(f"  ADHD:          {ucla_df['adhd'].sum()}")

=== ADHD-200 Cohort ===
Total subjects: 242
Cases (ADHD=1): 103
Controls (ADHD=0): 139

Age (mean ± std):
  Controls: 12.64 ± 2.82 (n=139)
  Cases: 13.42 ± 2.90 (n=103)

Sex distribution:
      Female (Sex=0)  Male (Sex=1)
ADHD                              
0                 57            82
1                 12            91


=== UCLA Clinical Cohort ===
Total subjects: 132
Cases (any disorder): 63
Controls: 69

Age (mean ± std):
  Controls: 30.25 ± 8.58 (n=69)
  Cases: 33.89 ± 9.48 (n=63)

Sex distribution:
           Female (Sex=0)  Male (Sex=1)
Diagnosis                              
0                      34            35
1                      20            43

Disorder-specific counts:
  Schizophrenia: 15
  Bipolar:       22
  ADHD:          26
